# AnnData Introduction Tutorial
Building and exploring AnnData objects from scratch.

## 1. Install & import

In [1]:
!pip install anndata scipy -q

import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import csr_matrix
print(ad.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 55.2 MB/s eta 0:00:00
0.12.10


/tmp/ipykernel_524/1710812499.py:7: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  print(ad.__version__)


## 2. Initialize AnnData
Create a basic AnnData object with sparse Poisson-distributed count data (100 cells x 2000 genes).

In [2]:
counts = csr_matrix(np.random.poisson(1, size=(100, 2000)), dtype=np.float32)
adata = ad.AnnData(counts)
adata

AnnData object with n_obs × n_vars = 100 × 2000

In [3]:
# Inspect the data matrix
adata.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 126409 stored elements and shape (100, 2000)>

In [4]:
# Set obs_names (cell barcodes) and var_names (gene IDs)
adata.obs_names = [f'Cell_{i:d}' for i in range(adata.n_obs)]
adata.var_names = [f'Gene_{i:d}' for i in range(adata.n_vars)]
print(adata.obs_names[:10])

Index(['Cell_0', 'Cell_1', 'Cell_2', 'Cell_3', 'Cell_4', 'Cell_5', 'Cell_6',
       'Cell_7', 'Cell_8', 'Cell_9'],
      dtype='object')


## 3. Subsetting AnnData
Subset by obs/var names, boolean masks, or integer indices — similar to pandas.

In [5]:
# Subset by name
adata[['Cell_1', 'Cell_10'], ['Gene_5', 'Gene_1900']]

View of AnnData object with n_obs × n_vars = 2 × 2

## 4. Adding observation-level metadata (adata.obs)
Both `adata.obs` and `adata.var` are pandas DataFrames.

In [6]:
ct = np.random.choice(['B', 'T', 'Monocyte'], size=(adata.n_obs,))
adata.obs['cell_type'] = pd.Categorical(ct)
adata.obs

,cell_type
Cell_0,Monocyte
Cell_1,Monocyte
Cell_2,Monocyte
Cell_3,B
Cell_4,B
...,...
Cell_95,Monocyte
Cell_96,Monocyte
Cell_97,T
Cell_98,Monocyte


In [7]:
# AnnData repr now shows obs
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'

In [8]:
# Subset using metadata
bdata = adata[adata.obs.cell_type == 'B']
bdata

View of AnnData object with n_obs × n_vars = 23 × 2000
    obs: 'cell_type'

## 5. Multi-dimensional embeddings (adata.obsm / adata.varm)
Store UMAP, PCA, t-SNE, etc. in `obsm`. Gene-level matrices go in `varm`.

In [9]:
adata.obsm['X_umap'] = np.random.normal(0, 1, size=(adata.n_obs, 2))
adata.varm['gene_stuff'] = np.random.normal(0, 1, size=(adata.n_vars, 5))
print(adata.obsm)
adata

AxisArrays with keys: X_umap


AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    obsm: 'X_umap'
    varm: 'gene_stuff'

## 6. Unstructured metadata (adata.uns)
Store any free-form metadata — lists, dicts, parameters — in `uns`.

In [10]:
adata.uns['random'] = [1, 2, 3]
adata.uns

OrderedDict([('random', [1, 2, 3])])

## 7. Layers
Store alternative representations of X (e.g. raw, normalized, log-transformed) as layers.

In [11]:
adata.layers['log_transformed'] = np.log1p(adata.X)
adata

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'cell_type'
    uns: 'random'
    obsm: 'X_umap'
    varm: 'gene_stuff'
    layers: 'log_transformed'

In [12]:
# Convert a layer to a pandas DataFrame
adata.to_df(layer='log_transformed')

,Gene_0,Gene_1,Gene_2,Gene_3,Gene_4,Gene_5,Gene_6,Gene_7,Gene_8,Gene_9,...,Gene_1990,Gene_1991,Gene_1992,Gene_1993,Gene_1994,Gene_1995,Gene_1996,Gene_1997,Gene_1998,Gene_1999
Cell_0,0.000000,0.693147,0.693147,1.098612,0.000000,1.098612,1.098612,0.000000,1.098612,1.098612,...,0.693147,0.000000,1.098612,0.693147,0.693147,0.693147,0.693147,1.098612,1.386294,1.098612
Cell_1,0.693147,0.693147,0.000000,0.000000,1.386294,0.693147,0.000000,0.693147,1.609438,0.000000,...,0.693147,0.000000,0.693147,1.386294,0.693147,0.693147,0.693147,0.693147,0.693147,0.000000
Cell_2,0.000000,1.386294,0.000000,0.000000,0.693147,1.098612,1.386294,1.098612,0.693147,0.000000,...,1.098612,0.693147,0.000000,0.693147,1.098612,0.693147,0.693147,0.000000,0.693147,0.693147
Cell_3,1.609438,0.000000,0.693147,1.386294,1.098612,0.693147,0.000000,0.000000,0.693147,0.693147,...,0.693147,0.693147,0.693147,0.000000,0.000000,0.693147,0.000000,1.098612,0.000000,0.000000
Cell_4,0.000000,1.098612,0.693147,1.098612,0.693147,0.693147,0.693147,1.098612,0.693147,1.098612,...,0.693147,0.693147,0.693147,0.000000,1.386294,0.000000,0.693147,1.098612,1.098612,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Cell_95,0.000000,1.098612,0.693147,1.098612,0.000000,0.000000,1.098612,0.693147,0.693147,0.000000,...,1.098612,1.609438,0.000000,1.098612,0.693147,0.693147,0.000000,0.693147,0.000000,0.000000
Cell_96,0.000000,0.000000,1.098612,0.000000,0.000000,0.000000,0.693147,0.000000,0.000000,0.693147,...,0.693147,0.693147,0.693147,0.693147,0.693147,0.693147,0.693147,1.386294,0.000000,0.000000
Cell_97,0.000000,0.000000,0.693147,1.386294,0.000000,0.000000,1.609438,0.693147,0.693147,0.693147,...,0.000000,0.693147,0.000000,1.386294,0.693147,0.000000,0.000000,0.693147,0.693147,1.098612
Cell_98,0.000000,0.693147,0.693147,0.693147,0.000000,0.000000,0.000000,0.693147,0.000000,0.000000,...,1.609438,0.000000,0.693147,0.693147,0.693147,0.693147,0.693147,0.693147,0.693147,0.693147


## 8. Write to disk (h5ad format)
AnnData uses HDF5-based `.h5ad` as its native file format.

In [13]:
adata.write('my_results.h5ad', compression='gzip')
!h5ls 'my_results.h5ad'

/bin/bash: line 1: h5ls: command not found


## 9. Views and copies
Subsetting returns a **view** (no extra memory). Call `.copy()` to get an independent AnnData object.

In [14]:
# Add richer obs metadata
obs_meta = pd.DataFrame({
    'time_yr': np.random.choice([0, 2, 4, 8], adata.n_obs),
    'subject_id': np.random.choice(['subject 1', 'subject 2', 'subject 4', 'subject 8'], adata.n_obs),
    'instrument_type': np.random.choice(['type a', 'type b'], adata.n_obs),
    'site': np.random.choice(['site x', 'site y'], adata.n_obs),
}, index=adata.obs_names)

adata = ad.AnnData(adata.X, obs=obs_meta, var=adata.var)
print(adata)

AnnData object with n_obs × n_vars = 100 × 2000
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'


In [15]:
# This is a view — no new memory allocated
adata[:5, ['Gene_1', 'Gene_3']]

View of AnnData object with n_obs × n_vars = 5 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [16]:
# Get an actual copy
adata_subset = adata[:5, ['Gene_1', 'Gene_3']].copy()
adata_subset

AnnData object with n_obs × n_vars = 5 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site'

In [17]:
# Modifying a view's X in-place
print('Before:', adata[:3, 'Gene_1'].X.toarray().tolist())
adata[:3, 'Gene_1'].X = [0, 0, 0]
print('After: ', adata[:3, 'Gene_1'].X.toarray().tolist())

Before: [[1.0], [1.0], [3.0]]
After:  [[0.0], [0.0], [0.0]]


/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:630: FutureWarning: Setting element `.X` of view of `AnnData` object will obey copy-on-write semantics in the next minor release. 
  if self._handle_view_X_cow(value):
/tmp/ipykernel_524/687208696.py:3: ImplicitModificationWarning: Modifying `X` on a view results in data being overridden
  adata[:3, 'Gene_1'].X = [0, 0, 0]


In [18]:
# Modifying obs of a view auto-converts it to a real AnnData
adata_subset = adata[:3, ['Gene_1', 'Gene_2']]
print('Before modification:', adata_subset.is_view)
adata_subset.obs['foo'] = range(3)
print('After modification: ', adata_subset.is_view)
adata_subset

Before modification: True
After modification:  False


/tmp/ipykernel_524/2052293346.py:4: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_subset.obs['foo'] = range(3)


AnnData object with n_obs × n_vars = 3 × 2
    obs: 'time_yr', 'subject_id', 'instrument_type', 'site', 'foo'

In [19]:
# Subset using pandas boolean indexing
adata[adata.obs.time_yr.isin([2, 4])].obs.head()

,time_yr,subject_id,instrument_type,site
Cell_0,4,subject 2,type a,site y
Cell_3,4,subject 8,type a,site y
Cell_5,2,subject 1,type b,site x
Cell_7,4,subject 8,type a,site y
Cell_8,2,subject 8,type a,site x


## 10. Backed mode — partial reading of large files
Read only parts of a large `.h5ad` file into memory using `backed='r'`.

In [20]:
adata_backed = ad.read_h5ad('my_results.h5ad', backed='r')
print('Is backed:', adata_backed.isbacked)
print('File path:', adata_backed.filename)
adata_backed.file.close()
print('File closed.')

Is backed: True
File path: my_results.h5ad
File closed.
